In [ ]:
import os
import tiktoken
from litellm import completion
from uri_shortener import UriShortener
import re

In [ ]:
# Set your Gemini API key from Google AI Studio
os.environ["GEMINI_API_KEY"] = input("Enter your Gemini API Key: ")

# Route to Gemini 3 Flash
MODEL = "gemini/gemini-3-flash-preview" 

In [6]:
encoder = tiktoken.get_encoding("cl100k_base")
shortener = UriShortener()

In [7]:
# 1. A realistic RAG payload with heavy, complex URLs
rag_context = """
# Retrieved Context: Indian Statutory Framework

## Document A: Indian Contract Act, 1872
An agreement enforceable by law is a contract under [Section 2(h)](/IN/ACT/1872/9#section-2.clause-h.enforceability). 
The provisions governing voidable contracts and undue influence are explicitly detailed in [Section 16](/IN/ACT/1872/9#section-16.undue-influence.burden-of-proof) 
and [Section 19.A](/IN/ACT/1872/9#section-19A.power-to-set-aside-induced-contract).

## Document B: Digital Personal Data Protection (DPDP) Act, 2023
Data Fiduciaries must process personal data only for a lawful purpose based on consent per [Section 6](/IN/ACT/2023/26#section-6.consent-requirements). 
The absolute obligations regarding data erasure and withdrawal of consent are codified under [Section 6(4)](/IN/ACT/2023/26#section-6.subsection-4.withdrawal-consequences) 
and [Section 8](/IN/ACT/2023/26#section-8.obligations-of-data-fiduciary).

## User Query
Summarize the rules regarding the withdrawal of consent for data processing and how undue influence impacts contractual validity under Indian law. Cite your sources.
"""

In [8]:
# 2. Encode the payload
encoded_context = shortener.encode_text(rag_context)

# 3. Calculate the exact token difference
raw_tokens = len(encoder.encode(rag_context))
encoded_tokens = len(encoder.encode(encoded_context))

print("=== THE TOKEN PROBLEM ===")
print(f"Raw Context Tokens:     {raw_tokens}")
print(f"Encoded Context Tokens: {encoded_tokens}")
print(f"Savings:                {raw_tokens - encoded_tokens} tokens ({(raw_tokens - encoded_tokens)/raw_tokens*100:.1f}% reduction)")
print("\n=== WHAT THE LLM ACTUALLY SEES ===")
print(encoded_context)

=== THE TOKEN PROBLEM ===
Raw Context Tokens:     300
Encoded Context Tokens: 201
Savings:                99 tokens (33.0% reduction)

=== WHAT THE LLM ACTUALLY SEES ===

# Retrieved Context: Indian Statutory Framework

## Document A: Indian Contract Act, 1872
An agreement enforceable by law is a contract under [Section 2(h)](=1#1). 
The provisions governing voidable contracts and undue influence are explicitly detailed in [Section 16](=1#2) 
and [Section 19.A](=1#3).

## Document B: Digital Personal Data Protection (DPDP) Act, 2023
Data Fiduciaries must process personal data only for a lawful purpose based on consent per [Section 6](=2#1). 
The absolute obligations regarding data erasure and withdrawal of consent are codified under [Section 6(4)](=2#2) 
and [Section 8](=2#3).

## User Query
Summarize the rules regarding the withdrawal of consent for data processing and how undue influence impacts contractual validity under Indian law. Cite your sources.



In [11]:
print(f"Sending encoded context to {MODEL}...")

# 1. Generate Answer
# Call the LLM with a system message guiding its behavior
response = completion(
    model=MODEL,
    messages=[
        {
            "role": "system", 
            "content": (
                "You are an expert legal assistant. Analyze the provided context "
                "and answer the user query accurately. "
                "CRITICAL: When citing sources, you MUST use the exact compressed "
                "short-code URLs provided in the text (e.g., `=1`, `=1#1`) inside standard "
                "Markdown links, like [Source Label](=1#1). Do not invent or alter these codes."
            )
        },
        {
            "role": "user", 
            "content": encoded_context
        }
    ],
    temperature=0.1
)

llm_output = response.choices[0].message.content

print("\n=== 1. RAW LLM OUTPUT (WITH SHORT CODES) ===")
print(llm_output)

# 2. Hallucination Audit
hallucinations = shortener.find_hallucinated_codes(llm_output)
if hallucinations:
    print(f"\n⚠️ WARNING: LLM hallucinated these codes: {hallucinations}")
else:
    print("\n✅ AUDIT PASSED: No hallucinated short codes detected.")

# 3. Final Decode
final_output = shortener.decode_text(llm_output)

print("\n=== 2. FINAL DECODED OUTPUT (READY FOR FRONTEND) ===")
print(final_output)

16:00:30 - LiteLLM:WARNING: vertex_and_google_ai_studio_gemini.py:1168 - DeprecationWarning: `temperature`, `top_p`, and `top_k` continue to function for Gemini 3+ (gemini-3-flash-preview) but are planned for removal in a future release. Move sampling guidance into the `system` instructions instead.


Sending encoded context to gemini/gemini-3-flash-preview...

=== 1. RAW LLM OUTPUT (WITH SHORT CODES) ===
Under Indian law, the rules regarding data processing consent and the impact of undue influence on contracts are governed by the Digital Personal Data Protection (DPDP) Act and the Indian Contract Act, respectively.

### **Withdrawal of Consent for Data Processing**
Under the **Digital Personal Data Protection (DPDP) Act, 2023**, Data Fiduciaries are required to process personal data only for lawful purposes and must obtain the individual's consent [Section 6](=2#1). The Act provides the following regarding withdrawal:
*   **Right to Withdraw:** The absolute obligations concerning the withdrawal of consent are codified under [Section 6(4)](=2#2).
*   **Consequences of Withdrawal:** Once consent is withdrawn, fiduciaries must adhere to statutory obligations regarding data erasure and the cessation of processing [Section 8](=2#3).

### **Undue Influence and Contractual Validity**
The

In [12]:
# Create a synthetic payload representing a heavy RAG retrieval (e.g., 50 chunks)
base_url = "/IN/ACT/2023/26#section-{}.subsection-{}.clause-{}"
heavy_rag_chunks = []

for i in range(1, 21):
    chunk = f"According to [Regulation {i}]({base_url.format(i, 1, 'a')}), compliance is mandatory. "
    chunk += f"Exceptions apply in [Regulation {i}b]({base_url.format(i, 2, 'b')}).\n"
    heavy_rag_chunks.append(chunk)

massive_payload = "\n".join(heavy_rag_chunks)

# Encode the massive payload
massive_encoded = shortener.encode_text(massive_payload)

# Calculate savings at scale
massive_raw_tokens = len(encoder.encode(massive_payload))
massive_encoded_tokens = len(encoder.encode(massive_encoded))
massive_savings = massive_raw_tokens - massive_encoded_tokens

print("=== WHY THIS MATTERS IN PRODUCTION ===")
print("Simulating a large RAG retrieval with 40 long URLs...\n")
print(f"Tokens without PromptLinks: {massive_raw_tokens}")
print(f"Tokens with PromptLinks:    {massive_encoded_tokens}")
print(f"Total Tokens Saved:         {massive_savings}")
print(f"Efficiency Gain:            {(massive_savings)/massive_raw_tokens*100:.1f}%")
print("\nInsight: In a typical 10,000 request/day system, this saves millions of tokens daily,")
print("slashes latency by reducing the prompt size, and completely removes the noisy text")
print("that distracts the LLM's attention mechanism.")

=== WHY THIS MATTERS IN PRODUCTION ===
Simulating a large RAG retrieval with 40 long URLs...

Tokens without PromptLinks: 1280
Tokens with PromptLinks:    640
Total Tokens Saved:         640
Efficiency Gain:            50.0%

Insight: In a typical 10,000 request/day system, this saves millions of tokens daily,
slashes latency by reducing the prompt size, and completely removes the noisy text
that distracts the LLM's attention mechanism.
